#Tokenizers for Finnish
In this notebook, we analyse and compare the tokenizers used by four pretrained transformer models for Finnish. We examine how differently they represent Finnish text and whether some tokenizers produce more efficient or consistent representations.

Tokenization is important because transformer models do not process raw text directly. Text is split into tokens and converted into numerical token IDs, which determine how the input is represented to the model. This is particularly relevant for Finnish, an agglutinative language where words can contain multiple grammatical and semantic components and become relatively long through suffixation. Finnish also uses compounding to create new words, which is why they can appear extremely long. Efficient tokenization can therefore help avoid unnecessary splitting into very small or uncommon subwords.

Finally, we compare the tokenizers' results. Better tokenization metrics do not necessarily lead to better model performance, but they can help explain differences in how the models represent Finnish text.

##Install dependencies
First install dependencies.

In [1]:
pip install transformers datasets evaluate pandas


## Load data
For this experiment, we use the [mteb/amazon_massive_intent](https://huggingface.co/datasets/mteb/amazon_massive_intent) dataset from Hugging Face. The dataset contains user utterances and their corresponding intent labels in multiple languages. We focus on the Finnish (fi) subset and use the same data later for fine-tuning the models, allowing us to compare tokenizer characteristics with downstream classification performance.

In [2]:
from datasets import load_dataset

dataset = load_dataset("mteb/amazon_massive_intent")

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

### Filter Finnish data

In [3]:
hf_train_dataset = dataset["train"].filter(lambda example: example['lang'] == 'fi')
hf_val_dataset = dataset["validation"].filter(lambda example: example['lang'] == 'fi')
hf_test_dataset = dataset["test"].filter(lambda example: example['lang'] == 'fi')

print(hf_train_dataset)
print(hf_val_dataset)
print(hf_test_dataset)

Dataset({
    features: ['id', 'label', 'label_text', 'text', 'lang'],
    num_rows: 11514
})
Dataset({
    features: ['id', 'label', 'label_text', 'text', 'lang'],
    num_rows: 2033
})
Dataset({
    features: ['id', 'label', 'label_text', 'text', 'lang'],
    num_rows: 2974
})


### Examples from the training set:

In [4]:
df_train = hf_train_dataset.to_pandas()
df_val = hf_val_dataset.to_pandas()
df_test = hf_test_dataset.to_pandas()

df_train.head()

,id,label,label_text,text,lang
0,1,alarm_set,alarm_set,herätä minut aamuyhdeksältä perjantaina,fi
1,2,alarm_set,alarm_set,laita hälytys kahden tunnin päähän tästä hetkestä,fi
2,4,audio_volume_mute,audio_volume_mute,olly hiljaa,fi
3,5,audio_volume_mute,audio_volume_mute,stop,fi
4,6,audio_volume_mute,audio_volume_mute,olly keskeytä kymmeneksi sekunniksi,fi


### Class distribution in the training set:

In [5]:
print(df_train["label"].value_counts())

label
calendar_set                810
play_music                  639
weather_query               573
calendar_query              566
general_quirky              555
qa_factoid                  544
news_query                  503
email_query                 418
email_sendemail             354
datetime_query              350
calendar_remove             312
social_post                 283
play_radio                  283
qa_definition               267
transport_query             227
cooking_recipe              207
lists_query                 198
play_podcasts               193
recommendation_events       190
alarm_set                   182
lists_createoradd           177
recommendation_locations    173
lists_remove                164
music_query                 154
iot_hue_lightoff            153
qa_stock                    152
play_audiobook              150
qa_currency                 142
takeaway_order              135
alarm_query                 130
transport_ticket            127
em

## Test tokenisers

Before tokenizing the full dataset, we first apply each tokenizer to a short sample to illustrate how the same Finnish text can be represented differently.

We compare four pretrained transformer models and their corresponding tokenizers:

- **FinBERT (TurkuNLP, 2019)**: A Finnish-specific BERT model using 50,000 WordPiece tokens as vocabulary. It follows the original BERT  base architecture, with 12 transformer layers, 12 attention heads, and a 512-token maximum sequence length. Its vocabulary was specifically developed to provide better Finnish word coverage than multilingual BERT.
- **Finnish RoBERTa (Finnish-NLP, 2020)**: A Finnish-specific RoBERTa model using Byte-Pair Encoding (BPE) with a vocabulary of 50,265 tokens. It is based on the RoBERTa architecture, which modifies the original BERT pretraining setup, including removing the next-sentence prediction. The model uses sequences of up to 512 tokens.
- **Finnish ModernBERT (TurkuNLP, 2025)**: A newer encoder architecture based on ModernBERT, which uses a vocabulary of 50,368 custom WordPiece tokens. It was pretrained on Finnish together with Swedish, English, code, Latin, and Northern Sámi. Compared with the older BERT/RoBERTa architectures, ModernBERT is designed for longer contexts and includes architectural changes such as rotary positional embeddings (RoPE) and alternating local and global attention. Its Finnish model can theoretically support much longer contexts than the 512-token limit of the older models.
- **XLM-RoBERTa (Facebook AI, 2020)**: A multilingual RoBERTa-based model trained on 94 languages. It uses SentencePiece-based subword tokenization with a vocabulary of around 250,000 tokens. Unlike the other three models, its tokenizer was designed to work across many languages rather than being optimized specifically for Finnish.

The Finnish-specific models are compared with XLM-R to investigate whether language-specific pretraining and tokenization result in more efficient representations of Finnish text.

### Define tokenizers

In [6]:
from transformers import AutoTokenizer

tokenizer_finbert = AutoTokenizer.from_pretrained("TurkuNLP/bert-base-finnish-cased-v1")
tokenizer_finnish_roberta = AutoTokenizer.from_pretrained("Finnish-NLP/roberta-large-finnish-v2")
tokenizer_finnish_modernbert = AutoTokenizer.from_pretrained("TurkuNLP/finnish-modernbert-base")
tokenizer_xlm_r = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

###Tokenise a short text

Let's check how the same Finnish text is represented by each tokenizer and how **tokens** differ from their corresponding **token IDs**.

After tokenization, a Hugging Face dataset typically contains fields such as:

- `input_ids` numerical token IDs representing the model's vocabulary
- `attention_mask` indicates which positions contain actual tokens (1) or padding (0)
- `token_type_ids` identifies which segment each token belongs to

We can convert the `input_ids` back into their token representations and compare how the different tokenizers split the same Finnish text.

In [7]:
text = "Suomen kieli on erittäin mielenkiintoinen kieli. Haluaisitko sinä oppia puhumaan suomea? Minä ainakin haluaisin!"

tokens_finbert = tokenizer_finbert.tokenize(
    text,
    add_special_tokens=True
)

tokens_finnish_roberta = tokenizer_finnish_roberta.tokenize(
    text,
    add_special_tokens=True
)

tokens_finnish_modernbert = tokenizer_finnish_modernbert.tokenize(
    text,
    add_special_tokens=True
)

tokens_xlm_r = tokenizer_xlm_r.tokenize(
    text,
    add_special_tokens=True
)

In [8]:
encoded_finbert = tokenizer_finbert(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

encoded_finnish_roberta = tokenizer_finnish_roberta(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

encoded_finnish_modernbert = tokenizer_finnish_modernbert(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

encoded_xlm_r = tokenizer_xlm_r(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

In [9]:
print("FinBERT tokens:", tokens_finbert)
print("FinBERT encoded information:", encoded_finbert)

FinBERT tokens: ['[CLS]', 'Suomen', 'kieli', 'on', 'erittäin', 'mielenkiinto', '##inen', 'kieli', '.', 'Halua', '##isitko', 'sinä', 'oppia', 'puhumaan', 'suomea', '?', 'Minä', 'ainakin', 'haluaisin', '!', '[SEP]']
FinBERT encoded information: {'input_ids': tensor([[  102,   663,  4187,   145,  1912,  4937,   187,  4187,   111,  7116,
         12271,  1908,  5631, 11137, 13984,   305,  2172,  1120,  6493,   380,
           103]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [10]:
print("Finnish RoBERTa tokens:", tokens_finnish_roberta)
print("Finnish RoBERTa encoded information:", encoded_finnish_roberta)

Finnish RoBERTa tokens: ['<s>', 'Suomen', 'Ġkieli', 'Ġon', 'ĠerittÃ¤in', 'Ġmielenkiintoinen', 'Ġkieli', '.', 'ĠHaluais', 'itko', 'ĠsinÃ¤', 'Ġoppia', 'Ġpuhumaan', 'Ġsuomea', '?', 'ĠMinÃ¤', 'Ġainakin', 'Ġhaluaisin', '!', '</s>']
Finnish RoBERTa encoded information: {'input_ids': tensor([[    0,  3987,  4275,   302,  1978,  9688,  4275,    18, 47045,  8875,
          2194,  5340, 13504, 16804,    35,  3106,  1501,  7345,     5,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [11]:
print("Finnish ModernBERT tokens:", tokens_finnish_modernbert)
print("Finnish ModernBERT encoded information:", encoded_finnish_modernbert)

Finnish ModernBERT tokens: ['[CLS]', 'Suomen', 'Ġkieli', 'Ġon', 'ĠerittÃ¤in', 'Ġmielenkiintoinen', 'Ġkieli', '.', 'ĠHal', 'ua', 'isitko', 'ĠsinÃ¤', 'Ġoppia', 'Ġpuhumaan', 'Ġsuomea', '?', 'ĠMinÃ¤', 'Ġainakin', 'Ġhaluaisin', '!', '[SEP]']
Finnish ModernBERT encoded information: {'input_ids': tensor([[    1,  4707, 18137,   349,  3802, 20726, 18137,    41,  4774,   681,
         22632,  4939, 11436, 28854, 35661,    58,  7863,  2802, 16071,    28,
             4]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [12]:
print("XLM-R tokens:", tokens_xlm_r)
print("XLM-R encoded information:", encoded_xlm_r)

XLM-R tokens: ['<s>', '▁Suomen', '▁kieli', '▁on', '▁erittäin', '▁mielenkiinto', 'inen', '▁kieli', '.', '▁Halu', 'ais', 'it', 'ko', '▁sinä', '▁oppi', 'a', '▁puhu', 'maan', '▁suo', 'mea', '?', '▁Minä', '▁ainakin', '▁haluaisi', 'n', '!', '</s>']
XLM-R encoded information: {'input_ids': tensor([[     0,  15340, 124440,     98,  51258, 157284,   2217, 124440,      5,
          86746,   1606,    217,    265, 104983,  30566,     11,  34448,   4044,
           4474,  37440,     32,  85958,  36910, 145313,     19,     38,      2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]])}


Notice how each tokenizer uses different strategies for representing subword boundaries:
- **TurkuNLP BERT:** `##` a continuation of the previous token (WordPiece)
- **Finnish RoBERTa:** `Ġ` represents a space before the token (BPE)
- **Finnish ModernBERT:** `Ġ` represents a space before the token (BPE)
- **XLM-R:** `▁` represents a space before the token (SentencePiece)

We can also observe differences in how morphologically rich words are tokenised, **"Haluaisitko"** becomes:
- **TurkuNLP BERT:** 'Halua', '##isitko'
- **Finnish RoBERTa:** 'ĠHaluais', 'itko'
- **Finnish ModernBERT:** 'ĠHal', 'ua', 'isitko'
- **XLM-R:** '▁Halu', 'ais', 'it', 'ko'

The models also use different special tokens to mark the beginning and end of the input:
- **TurkuNLP BERT:** `[CLS]`, `[SEP]`
- **Finnish RoBERTa:** `<s>`, `</s>`
- **Finnish ModernBERT:** `[CLS]`, `[SEP]`
- **XLM-R:** `<s>`, `</s>`

Finally, the output of some tokenizers may display characters such as **ä** and **ö** as Ã¤ and Ã¶.


## Tokenize dataset

We now tokenize the dataset to measure the tokenizers' behaviour on the Finnish data. For this analysis, we do not truncate or pad the inputs, allowing us to measure the true tokenization length of each text without imposing a model-specific maximum sequence length.


In [13]:
from datasets import Dataset

def tokenize_dataset(dataset, tokenizer):
    """
    Tokenizes the dataset WITHOUT truncation or padding.

    Use this dataset to analyze the tokenizer itself:
    - tokens per text
    - tokens per word
    - maximum sequence length
    - percentage of texts exceeding model limits
    - etc.

    Nothing is removed from the original text due to model length limits.
    """
    return dataset.map(
        lambda examples: tokenizer(
            examples["text"],
            truncation=False,
            padding=False
        ),
        batched=True
    )

### Tokenise with Finbert

In [14]:
# Tokenize the train data using FinBERT
tokenized_analysis_train_finbert = tokenize_dataset(hf_train_dataset, tokenizer_finbert)


In [15]:
# Convert input_ids from the example back to tokens
input_ids_finbert = tokenized_analysis_train_finbert[0]["input_ids"]
tokens_finbert = tokenizer_finbert.convert_ids_to_tokens(input_ids_finbert)

# Decode the whole sequence back to text
decoded_text_finbert = tokenizer_finbert.decode(input_ids_finbert)

print("Subword tokens for the first FinBERT training example:")
print(tokens_finbert)

print("\nToken IDs for the first FinBERT training example:")
print(input_ids_finbert)

print("\nDecoded text for the first FinBERT training example:")
print(decoded_text_finbert)

Subword tokens for the first FinBERT training example:
['[CLS]', 'herätä', 'minut', 'aamu', '##yhdeks', '##ältä', 'perjantaina', '[SEP]']

Token IDs for the first FinBERT training example:
[102, 17442, 4095, 5675, 43171, 2458, 2497, 103]

Decoded text for the first FinBERT training example:
[CLS] herätä minut aamuyhdeksältä perjantaina [SEP]


### Tokenise with Finnish Roberta

In [16]:
# Tokenize the train data using tokenizer_finnish_roberta
tokenized_analysis_train_finnish_roberta = tokenize_dataset(hf_train_dataset, tokenizer_finnish_roberta)

In [17]:
# Convert input_ids from the example back to tokens
input_ids_roberta = tokenized_analysis_train_finnish_roberta[0]["input_ids"]
tokens_roberta = tokenizer_finnish_roberta.convert_ids_to_tokens(input_ids_roberta)

# Decode the whole sequence back to text
decoded_text_roberta = tokenizer_finnish_roberta.decode(input_ids_roberta)

print("Subword tokens for the first Finnish RoBERTa training example:")
print(tokens_roberta)

print("\nToken IDs for the first Finnish RoBERTa training example:")
print(input_ids_roberta)

print("\nDecoded text for the first Finnish RoBERTa training example:")
print(decoded_text_roberta)

Subword tokens for the first Finnish RoBERTa training example:
['<s>', 'her', 'Ã¤tÃ¤', 'Ġminut', 'Ġaamu', 'yh', 'deks', 'Ã¤ltÃ¤', 'Ġperjantaina', '</s>']

Token IDs for the first Finnish RoBERTa training example:
[0, 2412, 40256, 4422, 6372, 564, 4647, 2226, 5567, 2]

Decoded text for the first Finnish RoBERTa training example:
<s>herätä minut aamuyhdeksältä perjantaina</s>


### Tokenise with Finnish Modernbert

In [18]:
# Tokenize the train data using tokenizer_finnish_modernbert
tokenized_analysis_train_finnish_modernbert = tokenize_dataset(hf_train_dataset, tokenizer_finnish_modernbert)

In [19]:
# Convert input_ids from the example back to tokens
input_ids_modernbert = tokenized_analysis_train_finnish_modernbert[0]["input_ids"]
tokens_modernbert = tokenizer_finnish_modernbert.convert_ids_to_tokens(input_ids_modernbert)

# Decode the whole sequence back to text
decoded_text_modernbert = tokenizer_finnish_modernbert.decode(input_ids_modernbert)

print("Subword tokens for the first Finnish Modernbert training example:")
print(tokens_modernbert)

print("\nToken IDs for the first Finnish Modernbert training example:")
print(input_ids_modernbert)

print("\nDecoded text for the first Finnish Modernbert training example:")
print(decoded_text_modernbert)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Subword tokens for the first Finnish Modernbert training example:
['[CLS]', 'her', 'Ã¤tÃ¤', 'Ġminut', 'Ġaamu', 'y', 'hde', 'ks', 'Ã¤ltÃ¤', 'Ġperjantaina', '[SEP]']

Token IDs for the first Finnish Modernbert training example:
[1, 874, 4696, 4701, 13880, 116, 2810, 345, 4405, 11460, 4]

Decoded text for the first Finnish Modernbert training example:
[CLS]herätä minut aamuyhdeksältä perjantaina[SEP]


### Tokenise with XLM-R

In [20]:
# Tokenize the train data using tokenizer_xlm_r
tokenized_analysis_train_xlm_r = tokenize_dataset(hf_test_dataset, tokenizer_xlm_r)

In [21]:
# Convert input_ids from the example back to tokens
input_ids_xlm_r = tokenized_analysis_train_xlm_r[0]["input_ids"]
tokens_xlm_r = tokenizer_xlm_r.convert_ids_to_tokens(input_ids_xlm_r)

print("Subword tokens for the first XLM-R training example:")
print(tokens_xlm_r)

print("\nToken IDs for the first XLM-R training example:")
print(input_ids_xlm_r)

# Decode the whole sequence back to text
decoded_text_xlm_r = tokenizer_xlm_r.decode(input_ids_xlm_r)
print("\nDecoded text for the first XLM-R training example:")
print(decoded_text_xlm_r)

Subword tokens for the first XLM-R training example:
['<s>', '▁her', 'ätä', '▁minut', '▁vi', 'id', 'eltä', '▁aamulla', '▁tällä', '▁viikolla', '</s>']

Token IDs for the first XLM-R training example:
[0, 604, 98897, 9160, 279, 532, 59172, 188537, 36585, 128062, 2]

Decoded text for the first XLM-R training example:
<s> herätä minut viideltä aamulla tällä viikolla</s>


# Analyse tokenizations
We calculate several statistics to compare the tokenization methods:

- total number of tokens
- average number of tokens per word
- number of singleton token types (tokens appearing only once)
- number of split words
- ...

The goal is not simply to minimise the number of tokens. Ideally, a tokenizer should represent text efficiently while retaining enough reusable subword units to represent a wide range of words. These subword units do not necessarily correspond to linguistically meaningful units, as they are learned from statistical patterns in the training data.


                 TOKENIZATION
                      │
          ┌───────────┼───────────┐
          ↓           ↓           ↓
      morphology   sequence    rare words
                   length
          │           │           │
          └───────────┼───────────┘
                      ↓
             text representation
                for the Transformer
                      │
                      ↓
             downstream performance

In [22]:
import pandas as pd
from collections import Counter
import re
import numpy as np


def analyze_tokenizer_metrics(hf_tokenized_dataset, tokenizer, tokenizer_name):
    """
    Analyze tokenizer behaviour on a tokenized Hugging Face dataset.

    Metrics include:
        - Total tokens, including special tokens
        - Total content tokens, excluding special tokens
        - Average tokens per original word
        - Median tokens per text
        - 95th percentile tokens per text
        - Maximum tokens per text
        - Percentage of texts over 128, 256, and 512 tokens
        - Unique token types
        - Singleton token types
        - Number and percentage of split words
        - Average subwords per split word
        - UNK token count and rate

    Notes:
        "Unique Tokens" = number of distinct token types observed.
        "Singleton Tokens" = token types that occur exactly once.

        UNK counts actual occurrences of the tokenizer's unknown token.
        It does not measure all words that are absent from the vocabulary,
        since subword tokenizers can represent unseen words using known
        subword units.
    """

    total_tokens_with_special_tokens = 0
    total_content_tokens = 0
    total_original_words_count = 0

    all_subword_tokens = []

    split_words_count = 0
    total_subwords_in_split_words = 0

    unk_token_count = 0

    # Store sequence lengths for each text.
    content_token_lengths = []

    unk_token = tokenizer.unk_token

    # Number of special tokens added to a single sequence.
    num_special_tokens = tokenizer.num_special_tokens_to_add(pair=False)

    for i in range(len(hf_tokenized_dataset)):

        tokenized_example = hf_tokenized_dataset[i]

        original_text = tokenized_example["text"]
        input_ids = tokenized_example["input_ids"]

        # ---------------------------------------------------------
        # Token counts
        # ---------------------------------------------------------

        total_tokens_with_special_tokens += len(input_ids)

        # Remove special tokens when measuring tokenizer behaviour.
        content_token_count = max(
            len(input_ids) - num_special_tokens,
            0
        )

        total_content_tokens += content_token_count
        content_token_lengths.append(content_token_count)

        # Convert token IDs to token strings.
        subword_tokens = tokenizer.convert_ids_to_tokens(input_ids)

        # Keep only content tokens for vocabulary statistics.
        content_tokens = subword_tokens[num_special_tokens:]

        all_subword_tokens.extend(content_tokens)

        # ---------------------------------------------------------
        # UNK tokens
        # ---------------------------------------------------------

        if unk_token:
            for token in content_tokens:
                if token == unk_token:
                    unk_token_count += 1

        # ---------------------------------------------------------
        # Original words
        # ---------------------------------------------------------

        original_words = re.findall(r'\S+', original_text)

        total_original_words_count += len(original_words)

        # ---------------------------------------------------------
        # Split words
        # ---------------------------------------------------------

        for word in original_words:

            if word:
                word_subtokens = tokenizer.tokenize(word)

                if len(word_subtokens) > 1:
                    split_words_count += 1
                    total_subwords_in_split_words += len(word_subtokens)

    # =============================================================
    # Calculate metrics
    # =============================================================

    # Average tokens per original whitespace-separated word.
    avg_tokens_per_word = (
        total_content_tokens / total_original_words_count
        if total_original_words_count > 0
        else 0
    )

    # Sequence-length statistics.
    median_tokens_per_text = (
        np.median(content_token_lengths)
        if content_token_lengths
        else 0
    )

    p95_tokens_per_text = (
        np.percentile(content_token_lengths, 95)
        if content_token_lengths
        else 0
    )

    max_tokens_per_text = (
        max(content_token_lengths)
        if content_token_lengths
        else 0
    )

    # -------------------------------------------------------------
    # Token frequency statistics
    # -------------------------------------------------------------

    token_frequencies = Counter(all_subword_tokens)

    # Number of distinct token types observed.
    unique_tokens = len(token_frequencies)

    # Number of token types occurring exactly once.
    singleton_tokens = sum(
        1
        for count in token_frequencies.values()
        if count == 1
    )

    # -------------------------------------------------------------
    # Word splitting statistics
    # -------------------------------------------------------------

    split_word_percentage = (
        100 * split_words_count / total_original_words_count
        if total_original_words_count > 0
        else 0
    )

    average_subwords_per_split_word = (
        total_subwords_in_split_words / split_words_count
        if split_words_count > 0
        else 0
    )

    # -------------------------------------------------------------
    # UNK statistics
    # -------------------------------------------------------------

    unk_token_rate = (
        100 * unk_token_count / total_content_tokens
        if total_content_tokens > 0
        else 0
    )

    return {
        "Total Tokens": total_tokens_with_special_tokens,
        "Total Content Tokens": total_content_tokens,
        "Average Tokens per Original Word": avg_tokens_per_word,

        "Median Tokens per Text": median_tokens_per_text,
        "95th Percentile Tokens per Text": p95_tokens_per_text,
        "Maximum Tokens per Text": max_tokens_per_text,

        "Unique Tokens": unique_tokens,
        "Singleton Tokens": singleton_tokens,

        "Number of Split Words": split_words_count,
        "Split Words (%)": split_word_percentage,
        "Average Subwords per Split Word": average_subwords_per_split_word,

        "UNK Token Count": unk_token_count,
        "UNK Token Rate (%)": unk_token_rate
    }


def compare_tokenizers_metrics(tokenizers_data):
    """
    Compare tokenization metrics across multiple tokenizers.

    Args:
        tokenizers_data:
            Dictionary where keys are tokenizer names and values are
            tuples of (hf_tokenized_dataset, tokenizer_object).

    Returns:
        Pandas DataFrame containing the comparison metrics.
    """

    results = {}

    for name, (hf_tokenized_dataset, tokenizer_obj) in tokenizers_data.items():

        results[name] = analyze_tokenizer_metrics(
            hf_tokenized_dataset,
            tokenizer_obj,
            name
        )

    return pd.DataFrame(results).T

In [23]:
# Prepare data for the new comparison function
tokenizers_to_compare_train = {
    "FinBERT": (tokenized_analysis_train_finbert, tokenizer_finbert),
    "Finnish RoBERTa": (tokenized_analysis_train_finnish_roberta, tokenizer_finnish_roberta),
    "Finnish ModernBERT": (tokenized_analysis_train_finnish_modernbert, tokenizer_finnish_modernbert),
    "XLM-R": (tokenized_analysis_train_xlm_r, tokenizer_xlm_r)
}

In [24]:
# Get and display the analysis table
analysis_df_train = compare_tokenizers_metrics(tokenizers_to_compare_train)
print("\nTokenization Analysis Results (Train Dataset):\n")
display(analysis_df_train)


Tokenization Analysis Results (Train Dataset):



,Total Tokens,Total Content Tokens,Average Tokens per Original Word,Median Tokens per Text,95th Percentile Tokens per Text,Maximum Tokens per Text,Unique Tokens,Singleton Tokens,Number of Split Words,Split Words (%),Average Subwords per Split Word,UNK Token Count,UNK Token Rate (%)
FinBERT,97293.0,74265.0,1.330223,6.0,12.0,32.0,8554.0,3837.0,13868.0,24.840137,2.329391,0.0,0.0
Finnish RoBERTa,103396.0,80368.0,1.439539,6.0,13.0,32.0,8968.0,3970.0,35325.0,63.273568,2.318075,0.0,0.0
Finnish ModernBERT,112888.0,89860.0,1.609558,7.0,14.0,41.0,7742.0,2947.0,41164.0,73.732290,2.428020,0.0,0.0
XLM-R,30659.0,24711.0,1.732768,8.0,15.0,64.0,3453.0,1496.0,7414.0,51.987939,2.409496,0.0,0.0


##Morphologically rich words

The quantitative results show that the tokenizers differ substantially in how frequently they split Finnish words. To better understand these differences, we also examine how morphologically rich Finnish words are divided into subword units.

Since grammatical and semantic information can often be expressed by adding multiple suffixes to a word stem, we examine examples of longer or heavily inflected Finnish words and compare their tokenization across the four models.

The goal is not to determine whether individual subwords correspond exactly to linguistic morphemes, since subword tokenizers learn statistical units rather than explicit morphological boundaries. Instead, we investigate whether the tokenizers produce reusable subword units that could help models represent related word forms and generalize to less frequent forms.

###Find most split words

In [25]:
from collections import defaultdict

def find_most_split_words(dataset, tokenizers, top_n=50):
    """
    Find words that are split into the most subwords by each tokenizer.

    tokenizers:
        Dictionary such as:
        {
            "FinBERT": tokenizer_finbert,
            "Finnish RoBERTa": tokenizer_finnish_roberta,
            "Finnish ModernBERT": tokenizer_finnish_modernbert,
            "XLM-R": tokenizer_xlm_r
        }
    """

    # Collect unique whitespace-separated words from the dataset
    words = set()

    for text in dataset["text"]:
        words.update(text.split())

    results = []

    for word in words:

        # Remove simple punctuation around the word
        clean_word = word.strip(".,!?;:\"'()[]")

        if not clean_word:
            continue

        row = {"word": clean_word}

        for tokenizer_name, tokenizer in tokenizers.items():
            tokens = tokenizer.tokenize(clean_word)

            row[tokenizer_name] = len(tokens)

        results.append(row)

    # Sort by the maximum number of subwords produced
    results.sort(
        key=lambda x: max(
            x[name] for name in tokenizers.keys()
        ),
        reverse=True
    )

    return results[:top_n]

In [26]:
tokenizers = {
    "FinBERT": tokenizer_finbert,
    "Finnish RoBERTa": tokenizer_finnish_roberta,
    "Finnish ModernBERT": tokenizer_finnish_modernbert,
    "XLM-R": tokenizer_xlm_r
}

most_split_words = find_most_split_words(
    hf_train_dataset,
    tokenizers,
    top_n=50
)

pd.DataFrame(most_split_words)

,word,FinBERT,Finnish RoBERTa,Finnish ModernBERT,XLM-R
0,sadasviideskymmenestuhanneskahdessadasseitsemä...,15,15,17,17
1,tuhatyhdeksänsataayhdeksänkymmentäyhdeksästä,9,11,16,14
2,tuhatyhdeksänsataakahdeksankymmentäneljä,10,11,16,13
3,yhdeksänsataayhdeksänkymmentäyhdeksän,7,8,14,10
4,tuhatyhdeksänsataayhdeksänkymmenestä,8,9,13,12
5,viisituhattaneljäsataakolmekymmentäyksi,11,10,13,11
6,tuhatyhdeksänsataakuusikymmentäkahdeksan,9,9,13,12
7,tuhatyhdeksänsataayhdeksänkymmentä,7,8,12,11
8,tuhatyhdeksänsataaseitsemänkymmentäkaksi,9,9,12,12
9,yhdeksänsataaseitsemänkymmentäkolme,8,7,11,9


###Analyse some samples
Let's compare the 5 manually picked words to see how they are tokenized. We observe that the majority of the words are numbers, which is why cherry-picking a couple of semantically different words may be more interesting.

In [27]:
def analyze_word(word):
    print("=" * 60)
    print(f"Word: {word}")

    tokenizations = {}

    for name, tokenizer in tokenizers.items():
        tokens = tokenizer.tokenize(word)
        tokenizations[name] = tokens

    counts = [len(tokens) for tokens in tokenizations.values()]
    difference = max(counts) - min(counts)

    print(f"Difference: {difference} subwords")

    for name, tokens in tokenizations.items():
        print(f"{name}: {tokens}")

In [28]:
analyze_word("tuhatyhdeksänsataakahdeksankymmentäneljä")

Word: tuhatyhdeksänsataakahdeksankymmentäneljä
Difference: 6 subwords
FinBERT: ['tuhat', '##yhdeks', '##än', '##sataa', '##kahd', '##eks', '##an', '##kymmentä', '##nel', '##jä']
Finnish RoBERTa: ['tu', 'hat', 'yh', 'deksÃ¤n', 'sataa', 'kah', 'deksan', 'kymmen', 'tÃ¤n', 'el', 'jÃ¤']
Finnish ModernBERT: ['tu', 'hat', 'y', 'hde', 'ks', 'Ã¤nsa', 'taa', 'ka', 'hde', 'ks', 'ank', 'ym', 'ment', 'Ã¤n', 'el', 'jÃ¤']
XLM-R: ['▁tu', 'hat', 'yh', 'deks', 'än', 'sata', 'a', 'kah', 'deks', 'an', 'kymmentä', 'nel', 'jä']


In [29]:
analyze_word("pähkinäsuklaapatukoille")

Word: pähkinäsuklaapatukoille
Difference: 4 subwords
FinBERT: ['pähkin', '##äs', '##uklaa', '##pat', '##ukoille']
Finnish RoBERTa: ['pÃ¤hkin', 'Ã¤s', 'uklaa', 'pat', 'uk', 'oille']
Finnish ModernBERT: ['p', 'Ã¤hkin', 'Ã¤s', 'uk', 'l', 'aap', 'at', 'uko', 'ille']
XLM-R: ['▁pä', 'hk', 'inä', 'suk', 'la', 'apa', 'tuko', 'ille']


In [30]:
analyze_word("iltakävelyrutiinini")

Word: iltakävelyrutiinini
Difference: 3 subwords
FinBERT: ['ilta', '##kävel', '##yr', '##uti', '##inin', '##i']
Finnish RoBERTa: ['ilta', 'kÃ¤vel', 'yr', 'utiin', 'ini']
Finnish ModernBERT: ['ilta', 'kÃ¤', 'vel', 'yr', 'u', 'tiin', 'ini']
XLM-R: ['▁il', 'ta', 'kä', 've', 'ly', 'ruti', 'inin', 'i']


In [31]:
analyze_word("suosikkilatinokappaleeni")

Word: suosikkilatinokappaleeni
Difference: 3 subwords
FinBERT: ['suosikki', '##lat', '##ino', '##kappaleen', '##i']
Finnish RoBERTa: ['suo', 's', 'ikk', 'ilat', 'ino', 'kappal', 'eeni']
Finnish ModernBERT: ['su', 'os', 'ikk', 'ilat', 'ino', 'kapp', 'al', 'eeni']
XLM-R: ['▁suosikki', 'la', 'tino', 'kappale', 'eni']


In [32]:
analyze_word("vaihtoehtorockmusiikista")

Word: vaihtoehtorockmusiikista
Difference: 4 subwords
FinBERT: ['vaihtoehto', '##rock', '##musiik', '##ista']
Finnish RoBERTa: ['vaihtoehto', 'rock', 'musi', 'ikista']
Finnish ModernBERT: ['vaihto', 'eh', 'tor', 'ock', 'm', 'usi', 'ik', 'ista']
XLM-R: ['▁vaihtoehto', 'rock', 'musi', 'ik', 'ista']


##Conclusion

The tokenizations show clear differences between the four models.

* **Splitting:** Finnish ModernBERT splits words more frequently than the other tokenizers. Approximately 73.7% of the whitespace-separated words are split into multiple subwords, compared with 63.3% for Finnish RoBERTa, 52.0% for XLM-R, and 24.8% for FinBERT. ModernBERT also has the highest average number of tokens per original word.
* **Sequence lengths:** XLM-R produces the highest average number of tokens per original word and the longest sequences overall, with a median of 8 tokens and a 95th percentile of 15. Finnish ModernBERT follows with a median of 7 and a 95th percentile of 14. FinBERT produces the shortest sequences, while Finnish RoBERTa falls between the other models.
* **Unique & singleton tokens:** The number of unique and singleton tokens also differs between the models, with XLM-R having the lowest number. These values reflect the vocabulary and token distribution in this dataset, but should not be interpreted as direct measures of tokenizer quality.

The qualitative examples show that **more splitting does not necessarily mean more linguistically meaningful splitting**. This is particularly noticeable with Finnish ModernBERT, which often produces many small fragments that do not obviously correspond to Finnish morphological or lexical units. For example, *vaihtoehtorockmusiikista* is split into `vaihto`, `eh`, `tor`, `ock`, `m`, `usi`, `ik`, and `ista`, whereas FinBERT produces larger and more interpretable units such as `vaihtoehto`, `rock`, `musiik`, and `ista`. Similar patterns can be observed in other morphologically rich Finnish words.

This is important because Finnish is highly agglutinative, and reusable subword units could potentially help models generalize across related word forms. However, the usefulness of this decomposition depends on the units produced: excessive fragmentation may instead result in less coherent representations and longer sequences.

Therefore, while **Finnish ModernBERT's high split rate makes it an interesting candidate**, the tokenizer analysis alone does not tell us whether this strategy is beneficial for downstream performance. We test this in the next notebook by fine-tuning all four models on the same Finnish text classification task.
